source: https://github.com/sinanuozdemir/quick-start-guide-to-llms/

#Imports and Setups

In [1]:
%%capture
!pip install pinecone openai sentence-transformers tiktoken datasets

In [25]:
import hashlib
from datetime import datetime, timezone
from tqdm import tqdm
import logging

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

In [26]:
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

In [27]:
HF_EMBED_MODEL = "intfloat/e5-base-v2"
INDEX_NAME = "semantic-search-e2-v2"
NAMESPACE = "default"
SIMILARITY_METRIC = "cosine"

#Text Embedding

In [28]:
embedder = SentenceTransformer(HF_EMBED_MODEL)

# Get the embedding dimension from the model directly
VECTOR_DIM = embedder.get_sentence_embedding_dimension()

print(f"Embedder modes is Loaded {HF_EMBED_MODEL} (dim={VECTOR_DIM})")

def _encode(texts):
    """
    Generates sentence embeddings for a list of text strings using the SentenceTransformer model.

    Args:
        texts (list[str]): A list of text strings to encode.

    Returns:
        numpy.ndarray: A NumPy array containing the sentence embeddings.
                       The shape of the array will be (number_of_texts, embedding_dimension).
    """
    return embedder.encode(
        texts,
        batch_size=64,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

Embedder modes is Loaded intfloat/e5-base-v2 (dim=768)


In [34]:
def get_embeddings(texts):
    """
    Generates embeddings for a list of document/passage texts.

    Args:
        texts (list[str]): A list of text strings representing documents or passages.

    Returns:
        list[list[float]]: A list of lists, where each inner list is the embedding
                           vector for the corresponding text string.
    """
    prefixed = [f"passage: {t}" for t in texts]
    return _encode(prefixed).tolist()

def get_embedding(text):
    """
    Generates a single embedding for a given text string.

    Args:
        text (str): The text string to embed.

    Returns:
        list[float]: The embedding vector for the input text.
    """
    return get_embeddings([text])[0]

In [35]:
print(get_embeddings(["hi", "hello"]))

[[0.0036866366863250732, 0.002383303828537464, -0.024172019213438034, -0.0186981912702322, 0.025343824177980423, -0.03734089806675911, 0.021805372089147568, 0.022332508116960526, -0.01589624211192131, -0.017240295186638832, -0.03721018135547638, 0.011311892420053482, -0.05127989500761032, 0.013576695695519447, -0.044337641447782516, 0.014696372672915459, 0.05256711319088936, -0.009317317977547646, 0.014005087316036224, -0.015161820687353611, -0.025974014773964882, -0.00935361534357071, 0.04005654901266098, -0.02685779146850109, -0.0008346582762897015, 0.018323630094528198, -0.014235707931220531, 0.04119637608528137, -0.07038631290197372, -0.03767559304833412, 0.021858863532543182, 0.0359891839325428, 0.06882542371749878, -0.04205596074461937, -0.06054433435201645, 0.016638515517115593, -0.051477789878845215, -0.011979619972407818, -0.08366627246141434, -0.015095623210072517, -0.02124502882361412, -0.021106738597154617, -0.03819377347826958, 0.018379516899585724, -0.04244999960064888, -

#Pinecone setups

In [29]:
from google.colab import userdata
pinecone_key = userdata.get("PINECONE_API_KEY")

pc = Pinecone(
    api_key=pinecone_key
)

In [33]:
# Create new index at the correct dimension if missing
if INDEX_NAME not in pc.list_indexes().names():
    print(f"Creating index {INDEX_NAME} (dim={VECTOR_DIM})\n")

    pc.create_index(
        name=INDEX_NAME,
        dimension=VECTOR_DIM,
        metric=SIMILARITY_METRIC,
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(name=INDEX_NAME)
index.describe_index_stats()

Creating index semantic-search-e2-v2 (dim=768)



{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}

In [36]:
def my_hash(s: str) -> str:
    """Return the MD5 hash of the input string as a hexadecimal string"""
    return hashlib.md5(s.encode()).hexdigest()

my_hash('I love to hash it')

'ae76cc4dfd345ecaeea9b8ba0d5c3437'

In [37]:
def prepare_for_pinecone(texts):
    """
    Prepares data for ingestion into Pinecone, generating unique IDs, embeddings,
    and metadata for a list of text strings.

    Args:
        texts (list[str]): A list of text strings to prepare for Pinecone.

    Returns:
        list[tuple]: A list of tuples, where each tuple contains:
                     - A unique ID (MD5 hash of the text).
                     - The embedding vector for the text.
                     - A dictionary containing metadata (original text and upload date).
    """
    now = datetime.now(timezone.utc).isoformat()

    # Generate vector embeddings for each string in the input list using the specified engine
    embeddings = get_embeddings(texts)

    return [
        (
            my_hash(text),  # Unique ID (MD5 hash) for each text string
            embedding,  # Vector embedding of the text string
            {"text": text, "date_uploaded": now, "model": HF_EMBED_MODEL}  # Metadata dictionary
        )

        for text, embedding in zip(texts, embeddings)  # Iterate over each text and its corresponding embedding
    ]

In [38]:
texts = ['hi']

_id, embedding, metadata = prepare_for_pinecone(texts)[0]

print('ID:  ',_id, '\nLEN: ', len(embedding), '\nMETA:', metadata)

ID:   49f68a5c8493ec2c0bf489821c21fc3b 
LEN:  768 
META: {'text': 'hi', 'date_uploaded': '2025-10-22T18:45:52.639336+00:00', 'model': 'intfloat/e5-base-v2'}


In [39]:
def upload_texts_to_pinecone(texts, namespace=NAMESPACE, batch_size=None, show_progress_bar=False):
    """
    Uploads a list of text strings to a Pinecone index in batches.

    Args:
        texts (list[str]): A list of text strings to upload.
        namespace (str, optional): The Pinecone namespace to upload to. Defaults to NAMESPACE.
        batch_size (int, optional): The number of texts to process in each batch.
                                    If None, all texts are uploaded in a single batch. Defaults to None.
        show_progress_bar (bool, optional): Whether to display a progress bar during the upload.
                                            Defaults to False.

    Returns:
        int: The total number of vectors successfully upserted into the Pinecone index.
    """
    # If batch_size is not specified, set it to the total number of texts
    if not batch_size:
        batch_size = len(texts)

    total_upserted = 0

    _range = range(0, len(texts), batch_size)
    # Use tqdm for a progress bar if show_progress_bar is True
    iterator = tqdm(_range) if show_progress_bar else _range

    # Iterate through the texts in batches
    for i in iterator:
        # Get the current batch of texts
        batch = texts[i: i + batch_size]
        # Prepare the batch for Pinecone (generate IDs, embeddings, and metadata)
        prepared = prepare_for_pinecone(batch)
        # Upsert the prepared vectors to the Pinecone index
        resp = index.upsert(vectors=prepared, namespace=namespace)
        # Add the number of upserted vectors from the response to the total count
        total_upserted += resp.get("upserted_count", len(prepared))

    # Return the total number of upserted vectors
    return total_upserted

In [40]:
def query_from_pinecone(query, top_k=3, include_metadata=True):
    """
    Embeds a query with the 'query: ' prefix (as required by the E5 model) and
    searches the Pinecone index for the most similar vectors.

    Args:
        query (str): The text string to use as the query.
        top_k (int, optional): The number of top most similar results to return. Defaults to 3.
        include_metadata (bool, optional): Whether to include the metadata
                                           (original text, date uploaded, etc.) in the results.
                                           Defaults to True.

    Returns:
        list[dict]: A list of dictionaries, where each dictionary represents a matching vector
                    and contains its ID, score, and optionally metadata.
    """
    # Embed the query using the _encode function, adding the "query: " prefix.
    # Get the first (and only) embedding and convert it to a list.
    qvec = _encode([f"query: {query}"])[0].tolist()
    # Query the Pinecone index with the embedded query vector.
    # Specify the number of top results (top_k), namespace, and whether to include metadata.
    # Return the list of matching results, defaulting to an empty list if no matches are found.
    return index.query(
        vector=qvec,
        top_k=top_k,
        namespace=NAMESPACE,
        include_metadata=include_metadata
    ).get("matches", [])

def delete_texts_from_pinecone(texts, namespace=NAMESPACE):
    """
    Deletes vectors from the Pinecone index based on a list of original text strings.

    Args:
        texts (list[str]): A list of original text strings whose corresponding vectors
                           should be deleted from the index.
        namespace (str, optional): The Pinecone namespace from which to delete the vectors.
                                   Defaults to NAMESPACE.

    Returns:
        object: The response object from the Pinecone delete operation.
    """
    # Compute the unique IDs (hashes) for each text string in the input list.
    ids = [my_hash(t) for t in texts]
    # Call the Pinecone index's delete method with the list of IDs and the namespace.
    return index.delete(ids=ids, namespace=namespace)

#Dataset Prepration

**Google-research/Xtreme**

The Cross-lingual TRansfer Evaluation of Multilingual Encoders (XTREME) benchmark is a benchmark for the evaluation of the cross-lingual generalization ability of pre-trained multilingual models. It covers 40 typologically diverse languages (spanning 12 language families) and includes nine tasks that collectively require reasoning about different levels of syntax and semantics. The languages in XTREME are selected to maximize language diversity, coverage in existing tasks, and availability of training data. Among these are many under-studied languages, such as the Dravidian languages Tamil (spoken in southern India, Sri Lanka, and Singapore), Telugu and Malayalam (spoken mainly in southern India), and the Niger-Congo languages Swahili and Yoruba, spoken in Africa.

In [41]:
from datasets import load_dataset

dataset = load_dataset("xtreme", "MLQA.en.en")

# rename test -> train and val -> test (as we will use it in later in this chapter)
dataset['train'] = dataset['test']
dataset['test'] = dataset['validation']
del dataset['validation']

dataset

DatasetDict({
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 1148
    })
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11590
    })
})

In [42]:
print("Test Q/A 1:\n")
dataset['train'][0]

Test Q/A 1:



{'id': 'a4968ca8a18de16aa3859be760e43dbd3af3fce9',
 'title': 'Area 51',
 'context': 'In 1994, five unnamed civilian contractors and the widows of contractors Walter Kasza and Robert Frost sued the USAF and the United States Environmental Protection Agency. Their suit, in which they were represented by George Washington University law professor Jonathan Turley, alleged they had been present when large quantities of unknown chemicals had been burned in open pits and trenches at Groom. Biopsies taken from the complainants were analyzed by Rutgers University biochemists, who found high levels of dioxin, dibenzofuran, and trichloroethylene in their body fat. The complainants alleged they had sustained skin, liver, and respiratory injuries due to their work at Groom, and that this had contributed to the deaths of Frost and Kasza. The suit sought compensation for the injuries they had sustained, claiming the USAF had illegally handled toxic materials, and that the EPA had failed in its duty t

In [43]:
#lets upload our dataset to pinecone
unique_passages = list(set(dataset["test"]["context"]))
for idx in tqdm(range(0, len(unique_passages), 32)):
    passages = unique_passages[idx: idx + 32]
    upload_texts_to_pinecone(passages, batch_size=len(passages))

#Checkout what hapened to our dataset
index.describe_index_stats()

100%|██████████| 31/31 [31:44<00:00, 61.45s/it]


{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'default': {'vector_count': 960}},
 'total_vector_count': 960,
 'vector_type': 'dense'}

#Test our embedding system and its informaiton retrival porcess

In [44]:
query_from_pinecone('Who analyzed the biopsies?')

[{'id': 'ccd4ad0d377c481cd374e61e2944d102',
  'metadata': {'date_uploaded': '2025-10-22T19:19:00.546984+00:00',
               'model': 'intfloat/e5-base-v2',
               'text': 'Body composition may be analyzed in various ways. This '
                       'can be done in terms of the chemical elements present, '
                       'or by molecular type e.g., water, protein, fats (or '
                       'lipids), hydroxylapatite (in bones), carbohydrates '
                       '(such as glycogen and glucose) and DNA. In terms of '
                       'tissue type, the body may be analyzed into water, fat, '
                       'connective tissue, muscle, bone, etc. In terms of cell '
                       'type, the body contains hundreds of different types of '
                       'cells, but notably, the largest number of cells '
                       'contained in a human body (though not the largest mass '
                       'of cells) are not human 

In [45]:
query_from_pinecone('Does an infection for Sandflies go away over time?')

[{'id': '2f90090e21f19450887d5f3ff781e541',
  'metadata': {'date_uploaded': '2025-10-22T19:16:55.679438+00:00',
               'model': 'intfloat/e5-base-v2',
               'text': 'Pappataci fever is prevalent in the subtropical zone of '
                       'the Eastern Hemisphere between 20°N and 45°N, '
                       'particularly in Southern Europe, North Africa, the '
                       'Balkans, Eastern Mediterranean, Iraq, Iran, Pakistan, '
                       'Afghanistan and India.The disease is transmitted by the '
                       'bites of phlebotomine sandflies of the Genus '
                       'Phlebotomus, in particular, Phlebotomus papatasi, '
                       'Phlebotomus perniciosus and Phlebotomus perfiliewi. The '
                       'sandfly becomes infected when biting an infected human '
                       'in the period between 48 hours before the onset of '
                       'fever and 24 hours after the end of t

#Cross Encoder
(Ranking simmilar vectors for finding the most related responses  retrives from pinecode vector database)

In [46]:
from sentence_transformers.cross_encoder import CrossEncoder
import numpy as np
from torch import nn
from copy import copy

In [47]:
def get_results_from_pinecone(query, top_k=3, re_rank_model=None, verbose=True, correct_hash=None):

    #Fetch most raleted results using cosine simmilarity between vecctors(query and saved records)
    results_from_pinecone = query_from_pinecone(query, top_k=top_k)

    #Handle no result ...
    if not results_from_pinecone:
        return []

    if verbose:
        print("Given Query:", query)


    final_results = []

    retrieved_correct_position, reranked_correct_position = None, None

    for idx, result_from_pinecone in enumerate(results_from_pinecone):
        if correct_hash and result_from_pinecone['id'] == correct_hash:
            retrieved_correct_position = idx

    if re_rank_model is not None:
        if verbose:
            print('Document ID (Hash)\t\tRetrieval Score\tCE Score\tText')

        sentence_combinations = [[query, result_from_pinecone['metadata']['text']] for result_from_pinecone in results_from_pinecone]

        # Compute the similarity scores for these combinations
        similarity_scores = re_rank_model.predict(sentence_combinations, activation_fct=nn.Sigmoid())

        # Sort the scores in decreasing order
        sim_scores_argsort = list(reversed(np.argsort(similarity_scores)))
        sim_scores_sort = list(reversed(np.sort(similarity_scores)))
        top_re_rank_score = sim_scores_sort[0]

        # Print the scores
        # print(list(zip(sim_scores_argsort, sim_scores_sort)))
        for idx, _ in enumerate(sim_scores_argsort):
            result_from_pinecone = results_from_pinecone[_]
            if correct_hash and retrieved_correct_position == _:
                reranked_correct_position = idx
            final_results.append({'score': similarity_scores[idx], 'id': result_from_pinecone['id'], 'metadata': result_from_pinecone['metadata']})
            if verbose:
                print(f"{result_from_pinecone['id']}\t{result_from_pinecone['score']:.2f}\t{similarity_scores[idx]:.6f}\t{result_from_pinecone['metadata']['text'][:50]}")
        return {'final_results': final_results, 'retrieved_correct_position': retrieved_correct_position, 'reranked_correct_position': reranked_correct_position, 'results_from_pinecone': results_from_pinecone, 'top_re_rank_score': top_re_rank_score}

    if verbose:
        print('Document ID (Hash)\t\tRetrieval Score\tText')
    for result_from_pinecone in results_from_pinecone:
        final_results.append(result_from_pinecone)
        if verbose:
            print(f"{result_from_pinecone['id']}\t{result_from_pinecone['score']:.2f}\t{result_from_pinecone['metadata']['text'][:50]}")

    return {
        'final_results': final_results,
        'retrieved_correct_position': retrieved_correct_position,
        'reranked_correct_position': reranked_correct_position
        }

In [48]:
# Pre-trained cross encoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2', num_labels=1)
cross_encoder

config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

CrossEncoder(
  (model): BertForSequenceClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 384, padding_idx=0)
        (position_embeddings): Embedding(512, 384)
        (token_type_embeddings): Embedding(2, 384)
        (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=384, out_features=384, bias=True)
                (key): Linear(in_features=384, out_features=384, bias=True)
                (value): Linear(in_features=384, out_features=384, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=384, out_features=38

In [49]:
#lets build our query right from our dataset!

q_to_hash = {data['question']: my_hash(data['context']) for data in dataset['test']}

unique_inputs = list(set(dataset['test']['question']))

query = unique_inputs[0]
print(query)

What year did Abel go to see his fiancee?


In [60]:
query_result = get_results_from_pinecone(
    query,
    top_k=1,
    re_rank_model=cross_encoder,
    correct_hash=q_to_hash[query],
    verbose=True
)

query_result

Given Query: What year did Abel go to see his fiancee?
Document ID (Hash)		Retrieval Score	CE Score	Text
e39cdc7bc51b757deac8be25f6e23b86	0.83	0.999574	While in Paris, Abel contracted tuberculosis. At C


{'final_results': [{'score': np.float32(0.99957365),
   'id': 'e39cdc7bc51b757deac8be25f6e23b86',
   'metadata': {'date_uploaded': '2025-10-22T19:17:58.279530+00:00',
    'model': 'intfloat/e5-base-v2',
    'text': 'While in Paris, Abel contracted tuberculosis. At Christmas 1828, he traveled by sled to Froland to visit his fiancée. He became seriously ill on the journey; and, although a temporary improvement allowed the couple to enjoy the holiday together, he died relatively soon after on 6 April 1829, just two days before a letter arrived from August Crelle. Crelle had been searching for a new job for Abel in Berlin and had actually managed to have him appointed as a Professor at the University of Berlin. Crelle wrote to Abel to tell him, but the good news came too late.'}}],
 'retrieved_correct_position': 0,
 'reranked_correct_position': 0,
 'results_from_pinecone': [{'id': 'e39cdc7bc51b757deac8be25f6e23b86',
   'metadata': {'date_uploaded': '2025-10-22T19:17:58.279530+00:00',
     

#Evaluation of our Cross Encoder Model

In [61]:
logger.setLevel(logging.CRITICAL)

TOP_K=10
test_sample = dataset['test']
predictions = []

for question in tqdm(test_sample['question']):
    results = get_results_from_pinecone(
        question,
        top_k=TOP_K,
        re_rank_model=cross_encoder,
        correct_hash=q_to_hash[question],
        verbose=False
    )

    predictions.append(results)

retrieved_accuracy = sum([prediction['retrieved_correct_position'] == 0 for prediction in predictions])/len(predictions)
re_ranked_accuracy = sum([prediction['reranked_correct_position'] == 0 for prediction in predictions])/len(predictions)

print(f'Accuracy without re-ranking: {retrieved_accuracy}')
print(f'Accuracy with re-ranking: {re_ranked_accuracy}')

100%|██████████| 1148/1148 [1:40:19<00:00,  5.24s/it]

Accuracy without re-ranking: 0.7517421602787456
Accuracy with re-ranking: 0.8353658536585366
